In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
# !pip install "numpy<2.0"
# !pip install torch==2.2.2 torchvision==0.17.2 monai
# !pip install scikit-image
# !pip install wandb
# !pip install import-ipynb
# !pip install nibabel 

In [ ]:
import os
import json
import time
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import monai
import torch
from tqdm.notebook import tqdm
import import_ipynb

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from monai.transforms import (
    LoadImaged,
    Compose,
    EnsureChannelFirstd,
    Spacingd,
    ResizeWithPadOrCropd,
    NormalizeIntensityd,
    RandFlipd,
    RandRotated,
    RandGaussianSmoothd,
)

from Functions import patients_dicts, par_voxelsize




In [ ]:
def split_dataset(dataset, per_train=16, per_val=4, seed=None):
    if seed is not None:
        random.seed(seed)

    categories = {}
    for patient in dataset:
        disease = patient["Disease"]
        categories.setdefault(disease, []).append(patient)

    split_train_dataset = []
    split_val_dataset = []

    for patient_ids in categories.values():
        random.shuffle(patient_ids)
        split_train_dataset.extend(patient_ids[:per_train])
        split_val_dataset.extend(patient_ids[per_train:per_train + per_val])

    return split_train_dataset, split_val_dataset


def format_number(value):
    if isinstance(value, int):
        return str(value)
    if value == 0:
        return "0"
    return f"{value:g}"




In [ ]:
data_path = "train"
data_path_test = "test"
runs_dir = "baseline/runs_lenient_single_1000"

os.makedirs(runs_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The used device is {device}")

seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

full_dict_list = patients_dicts(data_path)
test_dict_list = patients_dicts(data_path_test)

print("train patients:", len(full_dict_list))
print("test patients:", len(test_dict_list))


data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
])

train_dataset_raw = monai.data.Dataset(full_dict_list, transform=data_transform)

voxel_size = []
mean_voxel, std_voxel, max_voxelsize = par_voxelsize(voxel_size, train_dataset_raw)
voxel_volume = float(np.prod(mean_voxel))

print("Median voxel size:", mean_voxel)
print("Std voxel size:", std_voxel)
print("Max voxel size:", max_voxelsize)
print("Voxel volume:", voxel_volume)




In [ ]:
val_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(mean_voxel[0], mean_voxel[1], mean_voxel[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=(256, 256, 16)), #added cause the images needed to be devided by 16 for the strides in the Unet image
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
])

train_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(mean_voxel[0], mean_voxel[1], mean_voxel[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=(256, 256, 16)), #added cause the images needed to be devided by 16 for the strides in the Unet image we can look into if we can solve this an other way
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
    RandFlipd(keys=["imgED", "maskED", "imgES", "maskES"], prob=0.33, spatial_axis=[0, 1, 2]),
    RandRotated(
        keys=["imgED", "maskED", "imgES", "maskES"],
        range_x=0.4,
        prob=0.33,
        mode=["bilinear", "nearest", "bilinear", "nearest"],
    ),
    RandGaussianSmoothd(keys=["imgED", "imgES"], prob=0.33),
])

test_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(mean_voxel[0], mean_voxel[1], mean_voxel[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=(256, 256, 16)), #added cause the images needed to be devided by 16 for the strides in the Unet image
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
])

#using the split function to split the data
train_dict_list, val_dict_list = split_dataset(full_dict_list, per_train=16, per_val=4, seed=seed)

train_dataset = monai.data.Dataset(train_dict_list, transform=train_data_transform)
val_dataset = monai.data.Dataset(val_dict_list, transform=val_data_transform)
test_dataset = monai.data.Dataset(test_dict_list, transform=test_data_transform)

label_order = sorted(pd.Series([p["Disease"] for p in full_dict_list]).unique().tolist())
disease_label_order = [label for label in label_order if label != "NOR"]

print("train patients:", len(train_dict_list))
print("val patients:", len(val_dict_list))



In [ ]:
cfg = {
    "arch_name": "wider1",
    "channels": (32, 64, 128, 256, 512),
    "strides": (2, 2, 2, 2),
    "lr": 2e-3,
    "weight_decay": 1e-5,
    "batch_size": 2,
    "epochs": 1000,
    "loss_name": "DiceLoss",
    "min_epochs": 400,
    "early_stopping_patience": 200,
    "early_stopping_min_delta": 1e-4,
    "save_every_n_epochs": 10,
    "num_workers": 8,
    "visual_val_index": 0,
    "visual_test_index": 0,
}

cfg["run_name"] = (
    f'{cfg["arch_name"]}_{cfg["loss_name"]}_lr{format_number(cfg["lr"])}_'
    f'wd{format_number(cfg["weight_decay"])}_bs{cfg["batch_size"]}_'
    f'ep{cfg["epochs"]}_min{cfg["min_epochs"]}_pat{cfg["early_stopping_patience"]}'
)
run_dir = os.path.join(runs_dir, cfg["run_name"])

experiment_df = pd.DataFrame([cfg])
display(experiment_df)
print(f"Run folder: {run_dir}")
if os.path.exists(run_dir):
    print("Warning: this run folder already exists, so rerunning the same config will overwrite files in it.")



In [ ]:
use_wandb = False
wandb_project = "Deep_learning_project"

if use_wandb:
    import wandb
    wandb.login()




In [ ]:
#Defining the Unet model
def build_model(cfg):
    model = monai.networks.nets.UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=4, #since we want to segment 3 heart areas+ back
        channels=cfg["channels"], #checking if the chanels are correct
        strides=cfg["strides"],
        #num_res_units=2,
    ).to(device)
    return model


def build_loaders(cfg):
    train_loader = monai.data.DataLoader(
        train_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    val_loader = monai.data.DataLoader(
        val_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    test_loader = monai.data.DataLoader(
        test_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    return train_loader, val_loader, test_loader


def build_loss_function(cfg):
    if cfg["loss_name"] == "DiceLoss":
        return monai.losses.DiceLoss(softmax=True, to_onehot_y=True, batch=True)
    if cfg["loss_name"] == "DiceCELoss":
        return monai.losses.DiceCELoss(softmax=True, to_onehot_y=True, batch=True)
    raise ValueError(f'Unknown loss_name: {cfg["loss_name"]}')


def evaluate_model(model, loader, only_dice=False):
    model.eval()
    inferer = monai.inferers.SlidingWindowInferer(roi_size=[256, 256, 16])

    dice_metric = monai.metrics.DiceMetric(include_background=False, reduction="mean")
    dice_metric.reset()

    if not only_dice:
        hd95_metric = monai.metrics.HausdorffDistanceMetric(
            include_background=False,
            percentile=95,
            reduction="mean",
        )
        hd95_metric.reset()

    with torch.no_grad():
        for sample in loader:
            for phase in ["ED", "ES"]:
                image = sample[f"img{phase}"].to(device)
                mask = sample[f"mask{phase}"].to(device)

                output = inferer(image, network=model)
                model_mask = torch.argmax(output, dim=1)
                model_mask = torch.nn.functional.one_hot(model_mask, num_classes=4)
                model_mask = model_mask.permute(0, 4, 1, 2, 3).float()

                gt_mask = torch.nn.functional.one_hot(
                    mask.long().squeeze(1), num_classes=4
                ).permute(0, 4, 1, 2, 3).float()

                dice_metric(y_pred=model_mask, y=gt_mask)
                if not only_dice:
                    hd95_metric(y_pred=model_mask, y=gt_mask)

    dice = dice_metric.aggregate().item()
    if only_dice:
        return {"dice": float(dice), "hd95": None}

    hd95 = hd95_metric.aggregate().item()
    return {"dice": float(dice), "hd95": float(hd95)}


def average_loss(model, loader, loss_function):
    model.eval()
    loss_sum = 0.0
    steps = 0

    with torch.no_grad():
        for batch in loader:
            imagED = batch["imgED"].float().to(device)
            labelED = batch["maskED"].long().to(device)
            imagES = batch["imgES"].float().to(device)
            labelES = batch["maskES"].long().to(device)

            outputED = model(imagED)
            outputES = model(imagES)

            loss = (loss_function(outputED, labelED) + loss_function(outputES, labelES)) / 2
            loss_sum += loss.item()
            steps += 1

    return float(loss_sum / steps)


feature_cols = [
    "rv_ed", "myo_ed", "lv_ed",
    "rv_es", "myo_es", "lv_es",
    "rv_sv", "lv_sv", "rv_ef", "lv_ef",
    "myo_delta", "lv_rv_ratio_ed", "lv_rv_ratio_es",
    "myo_lv_ratio_ed", "myo_lv_ratio_es",
]


def volume_from_mask(mask, label, voxel_volume):
    return float((mask == label).sum() * voxel_volume)


def safe_ef(edv, esv):
    if edv <= 0:
        return np.nan
    return float((edv - esv) / edv)


def feature_row_from_masks(patient_id, disease, mask_ed, mask_es):
    rv_ed = volume_from_mask(mask_ed, 1, voxel_volume)
    myo_ed = volume_from_mask(mask_ed, 2, voxel_volume)
    lv_ed = volume_from_mask(mask_ed, 3, voxel_volume)
    rv_es = volume_from_mask(mask_es, 1, voxel_volume)
    myo_es = volume_from_mask(mask_es, 2, voxel_volume)
    lv_es = volume_from_mask(mask_es, 3, voxel_volume)

    return {
        "ID": patient_id,
        "Disease": disease,
        "rv_ed": rv_ed,
        "myo_ed": myo_ed,
        "lv_ed": lv_ed,
        "rv_es": rv_es,
        "myo_es": myo_es,
        "lv_es": lv_es,
        "rv_sv": rv_ed - rv_es,
        "lv_sv": lv_ed - lv_es,
        "rv_ef": safe_ef(rv_ed, rv_es),
        "lv_ef": safe_ef(lv_ed, lv_es),
        "myo_delta": myo_ed - myo_es,
        "lv_rv_ratio_ed": float(lv_ed / rv_ed) if rv_ed > 0 else np.nan,
        "lv_rv_ratio_es": float(lv_es / rv_es) if rv_es > 0 else np.nan,
        "myo_lv_ratio_ed": float(myo_ed / lv_ed) if lv_ed > 0 else np.nan,
        "myo_lv_ratio_es": float(myo_es / lv_es) if lv_es > 0 else np.nan,
    }


def predict_masks_for_sample(sample, model, inferer):
    with torch.no_grad():
        image_ed = sample["imgED"].unsqueeze(0).float().to(device)
        image_es = sample["imgES"].unsqueeze(0).float().to(device)
        pred_ed = torch.argmax(inferer(image_ed, network=model), dim=1).squeeze().cpu().numpy().astype(int)
        pred_es = torch.argmax(inferer(image_es, network=model), dim=1).squeeze().cpu().numpy().astype(int)
    return pred_ed, pred_es


def extract_pred_feature_table(dataset, model):
    rows = []
    inferer = monai.inferers.SlidingWindowInferer(roi_size=[256, 256, 16])
    model.eval()
    for sample in dataset:
        pred_ed, pred_es = predict_masks_for_sample(sample, model, inferer)
        rows.append(feature_row_from_masks(sample["ID"], sample["Disease"], pred_ed, pred_es))
    return pd.DataFrame(rows)


def fit_classifier(train_df):
    X_train = train_df[feature_cols].fillna(0.0)
    y_train = train_df["Disease"]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, multi_class="auto"))
    clf.fit(X_train, y_train)
    return clf


def evaluate_classifier_on_df(clf, split_df):
    X = split_df[feature_cols].fillna(0.0)
    y_true = split_df["Disease"]
    y_pred = clf.predict(X)

    recall_per_class = recall_score(y_true, y_pred, labels=label_order, average=None, zero_division=0)
    disease_recalls = [recall_value for label, recall_value in zip(label_order, recall_per_class) if label != "NOR"]

    stats = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "min_disease_recall": float(np.min(disease_recalls)) if len(disease_recalls) else np.nan,
    }

    for label, recall_value in zip(label_order, recall_per_class):
        stats[f"recall_{label}"] = float(recall_value)

    return stats


def cross_validate_classifier(feature_df, n_splits=5, seed=10):
    y = feature_df["Disease"].to_numpy()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_stats = []
    for train_idx, val_idx in skf.split(np.zeros(len(y)), y):
        fold_train_df = feature_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df = feature_df.iloc[val_idx].reset_index(drop=True)
        clf = fit_classifier(fold_train_df)
        fold_stats.append(evaluate_classifier_on_df(clf, fold_val_df))

    fold_stats_df = pd.DataFrame(fold_stats)
    cv_summary = {f"{col}_val": float(fold_stats_df[col].mean()) for col in fold_stats_df.columns}
    disease_recall_val_cols = [f"recall_{label}_val" for label in disease_label_order]
    if len(disease_recall_val_cols):
        cv_summary["min_disease_recall_val"] = float(min(cv_summary[col] for col in disease_recall_val_cols))
    return cv_summary


In [ ]:
def overlay_multiclass(ax, mask):
    mask = np.round(mask).astype(int)
    overlay = np.zeros((*mask.shape, 4), dtype=float)
    overlay[mask == 1] = [0, 1, 0, 0.8]
    overlay[mask == 2] = [1, 0, 0, 0.8]
    overlay[mask == 3] = [0, 0, 1, 0.8]
    ax.imshow(overlay, interpolation="nearest")


def print_run_config(cfg):
    print("\n" + "=" * 100)
    print(f"RUN: {cfg['run_name']}")
    print(
        f"arch={cfg['arch_name']} | loss={cfg['loss_name']} | lr={cfg['lr']} | "
        f"weight_decay={cfg['weight_decay']} | batch_size={cfg['batch_size']} | epochs={cfg['epochs']}"
    )
    print(
        f"channels={cfg['channels']} | strides={cfg['strides']} | "
        f"min_epochs={cfg['min_epochs']} | patience={cfg['early_stopping_patience']} | "
        f"save_every={cfg['save_every_n_epochs']}"
    )
    print("=" * 100)


def save_fixed_visual_snapshot(model, val_sample, test_sample, save_path, title_text):
    model.eval()
    inferer = monai.inferers.SlidingWindowInferer(roi_size=[256, 256, 16])

    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    sample_rows = [("val", val_sample, 0), ("test", test_sample, 1)]

    with torch.no_grad():
        for split_name, sample, row_idx in sample_rows:
            for phase_idx, phase in enumerate(["ED", "ES"]):
                image_tensor = sample[f"img{phase}"].unsqueeze(0).float().to(device)
                mask_tensor = sample[f"mask{phase}"].unsqueeze(0).long().to(device)

                output = inferer(image_tensor, network=model)
                model_mask = torch.argmax(output, dim=1)

                image = image_tensor.squeeze().cpu().numpy()
                gt_labels = mask_tensor.squeeze().cpu().numpy()
                model_labels = model_mask.squeeze().cpu().numpy()

                slice_idx = image.shape[-1] // 2
                col_offset = phase_idx * 3

                ax[row_idx, col_offset].imshow(image[:, :, slice_idx], cmap="gray")
                ax[row_idx, col_offset].set_title(f"{split_name} {sample['ID']} - {phase} image")

                ax[row_idx, col_offset + 1].imshow(image[:, :, slice_idx], cmap="gray")
                overlay_multiclass(ax[row_idx, col_offset + 1], gt_labels[:, :, slice_idx])
                ax[row_idx, col_offset + 1].set_title(f"{split_name} {sample['ID']} - {phase} ground truth")

                ax[row_idx, col_offset + 2].imshow(image[:, :, slice_idx], cmap="gray")
                overlay_multiclass(ax[row_idx, col_offset + 2], model_labels[:, :, slice_idx])
                ax[row_idx, col_offset + 2].set_title(f"{split_name} {sample['ID']} - {phase} prediction")

    for row in ax:
        for axis in row:
            axis.axis("off")

    fig.suptitle(title_text, fontsize=16)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_fixed_volume_snapshot(model, val_sample, test_sample, save_dir, title_text):
    model.eval()
    inferer = monai.inferers.SlidingWindowInferer(roi_size=[256, 256, 16])

    os.makedirs(save_dir, exist_ok=True)

    for split_name, sample in [("val", val_sample), ("test", test_sample)]:
        split_dir = os.path.join(save_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)
        split_metadata = {"sample_id": sample["ID"], "title": title_text, "phases": {}}

        with torch.no_grad():
            for phase in ["ED", "ES"]:
                image_tensor = sample[f"img{phase}"].unsqueeze(0).float().to(device)
                mask_tensor = sample[f"mask{phase}"].unsqueeze(0).long().to(device)
                output = inferer(image_tensor, network=model)
                pred_tensor = torch.argmax(output, dim=1)

                image = image_tensor.squeeze().cpu().numpy().astype(np.float32)
                gt_mask = mask_tensor.squeeze().cpu().numpy().astype(np.uint8)
                pred_mask = pred_tensor.squeeze().cpu().numpy().astype(np.uint8)

                np.savez_compressed(
                    os.path.join(split_dir, f"{phase}.npz"),
                    image=image,
                    gt_mask=gt_mask,
                    pred_mask=pred_mask,
                )

                split_metadata["phases"][phase] = {
                    "image_shape": list(image.shape),
                    "mask_shape": list(gt_mask.shape),
                }

        with open(os.path.join(split_dir, "metadata.json"), "w") as f:
            json.dump(split_metadata, f, indent=2)


fixed_val_sample = val_dataset[cfg["visual_val_index"]]
fixed_test_sample = test_dataset[cfg["visual_test_index"]]


def train_medmnistmodel(model, train_dataloader, val_dataloader, optimizer, cfg, run_dir):
    history_rows = []
    best_train_loss = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    loss_function = build_loss_function(cfg)

    epochs_without_improvement = 0
    stopped_early = False
    stop_epoch = None

    checkpoints_dir = os.path.join(run_dir, "checkpoints")
    visuals_dir = os.path.join(run_dir, "visual_snapshots")
    volumes_dir = os.path.join(run_dir, "volume_snapshots")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(visuals_dir, exist_ok=True)
    os.makedirs(volumes_dir, exist_ok=True)

    run = None
    if use_wandb:
        run = wandb.init(
            project=wandb_project,
            name=cfg["run_name"],
            config=cfg,
            reinit=True,
        )

    start_time = time.time()
    table_handle = None
    epoch_bar = tqdm(range(1, cfg["epochs"] + 1), desc=cfg["run_name"], position=1, leave=False)

    for epoch in epoch_bar:
        model.train()
        steps = 0
        epoch_loss = 0.0

        for batch in train_dataloader: #250 steps per epoch, figure out how many steps we need, going to the data once, early stopping if patient level =15
            optimizer.zero_grad() #backropagation
            imagED = batch["imgED"].float().to(device)
            labelsED = batch["maskED"].long().to(device)
            imagES = batch["imgES"].float().to(device)
            labelsES = batch["maskES"].long().to(device)

            outputED = model(imagED)
            outputES = model(imagES)

            loss = (loss_function(outputED, labelsED) + loss_function(outputES, labelsES)) / 2
            epoch_loss += loss.item()
            loss.backward()
            optimizer.step()
            steps += 1

        avg_train_loss = epoch_loss / steps

        val_steps = 0
        val_epoch_loss = 0.0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                imagED = batch["imgED"].float().to(device)
                labelED = batch["maskED"].long().to(device)
                imagES = batch["imgES"].float().to(device)
                labelES = batch["maskES"].long().to(device)

                outputED = model(imagED)
                outputES = model(imagES)

                loss = (loss_function(outputED, labelED) + loss_function(outputES, labelES)) / 2
                val_epoch_loss += loss.item()
                val_steps += 1

        avg_val_loss = val_epoch_loss / val_steps

        row = {
            "epoch": epoch,
            "loss_train": float(avg_train_loss),
            "loss_val": float(avg_val_loss),
            "elapsed_min": (time.time() - start_time) / 60.0,
        }

        improved = avg_val_loss < (best_val_loss - cfg["early_stopping_min_delta"])

        if improved:
            best_train_loss = avg_train_loss
            best_val_loss = avg_val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), os.path.join(run_dir, "best_model.pt"))
        else:
            if epoch >= cfg["min_epochs"]:
                epochs_without_improvement += 1
                if epochs_without_improvement >= cfg["early_stopping_patience"]:
                    print(
                        f"Early stopping at epoch {epoch} | "
                        f"best epoch: {best_epoch} | best val loss: {best_val_loss:.6f}"
                    )
                    stopped_early = True
                    stop_epoch = epoch
                    history_rows.append(row)
                    break

        history_rows.append(row)
        history_df = pd.DataFrame(history_rows)
        history_df.to_csv(os.path.join(run_dir, "history.csv"), index=False)

        if epoch % cfg["save_every_n_epochs"] == 0:
            torch.save(model.state_dict(), os.path.join(checkpoints_dir, f"epoch_{epoch:04d}.pt"))
            save_fixed_visual_snapshot(
                model,
                fixed_val_sample,
                fixed_test_sample,
                os.path.join(visuals_dir, f"epoch_{epoch:04d}.png"),
                f"epoch {epoch}",
            )
            save_fixed_volume_snapshot(
                model,
                fixed_val_sample,
                fixed_test_sample,
                os.path.join(volumes_dir, f"epoch_{epoch:04d}"),
                f"epoch {epoch}",
            )

        table_view = history_df[["epoch", "loss_train", "loss_val", "elapsed_min"]].tail(8)
        if table_handle is None:
            table_handle = display(table_view, display_id=True)
        else:
            table_handle.update(table_view)

        epoch_bar.set_postfix({
            "loss_train": f"{avg_train_loss:.4f}",
            "loss_val": f"{avg_val_loss:.4f}",
        })

        if use_wandb:
            wandb.log(row)

    if use_wandb:
        wandb.finish()

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(os.path.join(run_dir, "history.csv"), index=False)
    torch.save(model.state_dict(), os.path.join(run_dir, "last_model.pt"))

    if len(history_df) > 0:
        final_epoch = int(history_df["epoch"].iloc[-1])
        if final_epoch % cfg["save_every_n_epochs"] != 0:
            save_fixed_visual_snapshot(
                model,
                fixed_val_sample,
                fixed_test_sample,
                os.path.join(visuals_dir, f"epoch_{final_epoch:04d}_last.png"),
                f"epoch {final_epoch} (last)",
            )
            save_fixed_volume_snapshot(
                model,
                fixed_val_sample,
                fixed_test_sample,
                os.path.join(volumes_dir, f"epoch_{final_epoch:04d}_last"),
                f"epoch {final_epoch} (last)",
            )

    return history_df, best_epoch, float(best_train_loss), float(best_val_loss), stopped_early, stop_epoch




In [ ]:
def build_summary_row(cfg, best_epoch, train_time, seg_train, seg_val, seg_test, clf_train, clf_val, clf_test):
    summary = {
        "run_name": cfg["run_name"],
        "arch_name": cfg["arch_name"],
        "loss_name": cfg["loss_name"],
        "lr": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "batch_size": cfg["batch_size"],
        "channels": str(cfg["channels"]),
        "strides": str(cfg["strides"]),
        "epochs": cfg["epochs"],
        "best_epoch": best_epoch,
        "train_time": train_time,
        "loss_train": seg_train["loss"],
        "loss_val": seg_val["loss"],
        "loss_test": seg_test["loss"],
        "dice_train": seg_train["dice"],
        "dice_val": seg_val["dice"],
        "dice_test": seg_test["dice"],
        "hd95_train": seg_train["hd95"],
        "hd95_val": seg_val["hd95"],
        "hd95_test": seg_test["hd95"],
        "macro_f1_train": clf_train["macro_f1"],
        "macro_f1_val": clf_val["macro_f1_val"],
        "macro_f1_test": clf_test["macro_f1"],
        "min_disease_recall_train": clf_train["min_disease_recall"],
        "min_disease_recall_val": clf_val["min_disease_recall_val"],
        "min_disease_recall_test": clf_test["min_disease_recall"],
    }

    recall_label_order = disease_label_order + (["NOR"] if "NOR" in label_order else [])
    for label in recall_label_order:
        summary[f"recall_{label}_train"] = clf_train[f"recall_{label}"]
        summary[f"recall_{label}_val"] = clf_val[f"recall_{label}_val"]
        summary[f"recall_{label}_test"] = clf_test[f"recall_{label}"]

    return summary


def evaluate_experiment(cfg):
    run_dir = os.path.join(runs_dir, cfg["run_name"])
    model_path = os.path.join(run_dir, "best_model.pt")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No best_model.pt found for {cfg['run_name']} in {run_dir}")

    model = build_model(cfg)
    train_loader, val_loader, test_loader = build_loaders(cfg)
    loss_function = build_loss_function(cfg)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    seg_train = {"loss": average_loss(model, train_loader, loss_function), **evaluate_model(model, train_loader)}
    seg_val = {"loss": average_loss(model, val_loader, loss_function), **evaluate_model(model, val_loader)}
    seg_test = {"loss": average_loss(model, test_loader, loss_function), **evaluate_model(model, test_loader)}

    pred_train_df = extract_pred_feature_table(train_dataset, model)
    pred_test_df = extract_pred_feature_table(test_dataset, model)

    clf_val = cross_validate_classifier(pred_train_df, n_splits=5, seed=seed)
    clf = fit_classifier(pred_train_df)
    clf_train = evaluate_classifier_on_df(clf, pred_train_df)
    clf_test = evaluate_classifier_on_df(clf, pred_test_df)

    history_df = pd.read_csv(os.path.join(run_dir, "history.csv"))
    best_epoch = int(history_df.loc[history_df["loss_val"].idxmin(), "epoch"])
    train_time = float(history_df["elapsed_min"].iloc[-1])

    summary = build_summary_row(cfg, best_epoch, train_time, seg_train, seg_val, seg_test, clf_train, clf_val, clf_test)
    pd.DataFrame([summary]).to_csv(os.path.join(run_dir, "summary.csv"), index=False)
    return summary


def train_experiment(cfg):
    print(datetime.now())
    print_run_config(cfg)

    run_dir = os.path.join(runs_dir, cfg["run_name"])
    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visual_snapshots"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "volume_snapshots"), exist_ok=True)

    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump({**cfg, "channels": list(cfg["channels"]), "strides": list(cfg["strides"])}, f, indent=2)

    model = build_model(cfg)
    train_loader, val_loader, _ = build_loaders(cfg)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    history_df, best_epoch, best_train_loss, best_val_loss, stopped_early, stop_epoch = train_medmnistmodel(
        model, train_loader, val_loader, optimizer, cfg, run_dir
    )

    training_summary = {
        "run_name": cfg["run_name"],
        "arch_name": cfg["arch_name"],
        "loss_name": cfg["loss_name"],
        "lr": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "batch_size": cfg["batch_size"],
        "channels": str(cfg["channels"]),
        "strides": str(cfg["strides"]),
        "epochs": cfg["epochs"],
        "best_epoch": best_epoch,
        "best_train_loss": float(best_train_loss),
        "best_val_loss": float(best_val_loss),
        "train_time": float(history_df["elapsed_min"].iloc[-1]),
        "epochs_ran": int(history_df["epoch"].iloc[-1]),
        "stopped_early": bool(stopped_early),
        "stop_epoch": None if stop_epoch is None else int(stop_epoch),
        "min_epochs": cfg["min_epochs"],
        "early_stopping_patience": cfg["early_stopping_patience"],
        "early_stopping_min_delta": cfg["early_stopping_min_delta"],
        "save_every_n_epochs": cfg["save_every_n_epochs"],
    }

    pd.DataFrame([training_summary]).to_csv(os.path.join(run_dir, "training_summary.csv"), index=False)

    print("\nBest-checkpoint visualization on fixed validation and test samples")
    model.load_state_dict(torch.load(os.path.join(run_dir, "best_model.pt"), map_location=device))
    save_fixed_visual_snapshot(
        model,
        fixed_val_sample,
        fixed_test_sample,
        os.path.join(run_dir, "best_model_visual.png"),
        "best model",
    )
    save_fixed_volume_snapshot(
        model,
        fixed_val_sample,
        fixed_test_sample,
        os.path.join(run_dir, "best_model_volume"),
        "best model",
    )

    print(datetime.now())
    return training_summary, history_df


In [ ]:
training_summary, history_df = train_experiment(cfg)

pd.read_csv(os.path.join(run_dir, "training_summary.csv"))


In [ ]:
summary = evaluate_experiment(cfg)

pd.read_csv(os.path.join(run_dir, "summary.csv"))


In [ ]:
print("Saved run files inside:")
print(run_dir)
display(pd.DataFrame([training_summary]))
display(pd.DataFrame([summary]))


In [ ]:
history = pd.read_csv(os.path.join(run_dir, "history.csv"))

plt.figure(figsize=(8, 4))
plt.plot(history["epoch"], history["loss_train"], label="loss_train")
plt.plot(history["epoch"], history["loss_val"], label="loss_val")
plt.xlabel("epoch")
plt.legend()
plt.title(cfg["run_name"])
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "history_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

print(history.tail(10).to_string(index=False))


In [ ]:
checkpoint_files = sorted([p.name for p in Path(os.path.join(run_dir, "checkpoints")).glob("*.pt")])
visual_files = sorted([p.name for p in Path(os.path.join(run_dir, "visual_snapshots")).glob("*.png")])
volume_dirs = sorted([p.name for p in Path(os.path.join(run_dir, "volume_snapshots")).glob("*") if p.is_dir()])

print()
print(f"Checkpoint files saved: {len(checkpoint_files)}")
print(f"Visual snapshot files saved: {len(visual_files)}")
print(f"Volume snapshot folders saved: {len(volume_dirs)}")
print(f"First checkpoint files: {checkpoint_files[:5]}")
print(f"First visual files: {visual_files[:5]}")
print(f"First volume folders: {volume_dirs[:5]}")


In [ ]:
def parse_checkpoint_epoch(path_obj):
    stem = path_obj.stem
    if stem.startswith("epoch_"):
        epoch_text = stem.replace("epoch_", "").split("_")[0]
        if epoch_text.isdigit():
            return int(epoch_text)
    return None


def evaluate_model_checkpoint(cfg, model_path, epoch_value, checkpoint_label):
    model = build_model(cfg)
    train_loader, val_loader, test_loader = build_loaders(cfg)
    loss_function = build_loss_function(cfg)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    seg_train = {"loss": average_loss(model, train_loader, loss_function), **evaluate_model(model, train_loader)}
    seg_val = {"loss": average_loss(model, val_loader, loss_function), **evaluate_model(model, val_loader)}
    seg_test = {"loss": average_loss(model, test_loader, loss_function), **evaluate_model(model, test_loader)}

    pred_train_df = extract_pred_feature_table(train_dataset, model)
    pred_test_df = extract_pred_feature_table(test_dataset, model)

    clf_val = cross_validate_classifier(pred_train_df, n_splits=5, seed=seed)
    clf = fit_classifier(pred_train_df)
    clf_train = evaluate_classifier_on_df(clf, pred_train_df)
    clf_test = evaluate_classifier_on_df(clf, pred_test_df)

    summary_row = build_summary_row(
        cfg=cfg,
        best_epoch=epoch_value,
        train_time=np.nan,
        seg_train=seg_train,
        seg_val=seg_val,
        seg_test=seg_test,
        clf_train=clf_train,
        clf_val=clf_val,
        clf_test=clf_test,
    )
    summary_row["evaluated_epoch"] = int(epoch_value)
    summary_row["checkpoint_label"] = checkpoint_label
    summary_row["checkpoint_path"] = str(model_path)
    return summary_row


run_dir = os.path.join(runs_dir, cfg["run_name"])
summary_df = pd.read_csv(os.path.join(run_dir, "summary.csv"))
best_epoch = int(summary_df.loc[0, "best_epoch"])
checkpoints_dir = Path(run_dir) / "checkpoints"

checkpoint_candidates = []
for checkpoint_path in sorted(checkpoints_dir.glob("epoch_*.pt")):
    checkpoint_epoch = parse_checkpoint_epoch(checkpoint_path)
    if checkpoint_epoch is None:
        continue
    if checkpoint_epoch < best_epoch:
        checkpoint_candidates.append((checkpoint_epoch, checkpoint_path, f"checkpoint_epoch_{checkpoint_epoch:04d}"))

checkpoint_candidates = sorted(checkpoint_candidates, key=lambda x: x[0])
checkpoint_candidates.append((best_epoch, Path(run_dir) / "best_model.pt", "best_model"))

checkpoint_rows = []
table_handle = None
display_columns = [
    "evaluated_epoch", "checkpoint_label",
    "run_name", "arch_name", "loss_name", "lr", "weight_decay", "batch_size", "channels", "strides", "epochs", "best_epoch", "train_time",
    "loss_train", "loss_val", "loss_test",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
    "macro_f1_train", "macro_f1_val", "macro_f1_test",
    "min_disease_recall_train", "min_disease_recall_val", "min_disease_recall_test",
]

checkpoint_bar = tqdm(checkpoint_candidates, desc="Checkpoint evaluation")
for checkpoint_epoch, checkpoint_path, checkpoint_label in checkpoint_bar:
    checkpoint_bar.set_postfix({"epoch": checkpoint_epoch, "label": checkpoint_label})
    checkpoint_rows.append(
        evaluate_model_checkpoint(
            cfg=cfg,
            model_path=checkpoint_path,
            epoch_value=checkpoint_epoch,
            checkpoint_label=checkpoint_label,
        )
    )

    checkpoint_table_df = pd.DataFrame(checkpoint_rows)
    available_columns = [col for col in display_columns if col in checkpoint_table_df.columns]
    table_view = checkpoint_table_df[available_columns].tail(8)
    if table_handle is None:
        table_handle = display(table_view, display_id=True)
    else:
        table_handle.update(table_view)

checkpoint_results_df = pd.DataFrame(checkpoint_rows)
checkpoint_results_df = checkpoint_results_df.sort_values(["evaluated_epoch", "checkpoint_label"]).reset_index(drop=True)
checkpoint_results_path = os.path.join(run_dir, "checkpoint_evaluations.csv")
checkpoint_results_df.to_csv(checkpoint_results_path, index=False)

print(f"Saved checkpoint evaluation results to: {checkpoint_results_path}")
display(checkpoint_results_df)


In [ ]:
checkpoint_results_path = os.path.join(run_dir, "checkpoint_evaluations.csv")
checkpoint_results_df = pd.read_csv(checkpoint_results_path)
checkpoint_results_df = checkpoint_results_df.sort_values(["evaluated_epoch", "checkpoint_label"]).reset_index(drop=True)

metric_groups = [
    ("Segmentation Loss", ["loss_train", "loss_val", "loss_test"]),
    ("Segmentation Dice", ["dice_train", "dice_val", "dice_test"]),
    ("Segmentation HD95", ["hd95_train", "hd95_val", "hd95_test"]),
    ("Classification Macro F1", ["macro_f1_train", "macro_f1_val", "macro_f1_test"]),
    ("Classification Min Disease Recall", ["min_disease_recall_train", "min_disease_recall_val", "min_disease_recall_test"]),
]

fig, axes = plt.subplots(len(metric_groups), 1, figsize=(12, 4 * len(metric_groups)), sharex=True)
if len(metric_groups) == 1:
    axes = [axes]

for ax, (title, columns) in zip(axes, metric_groups):
    for col in columns:
        ax.plot(checkpoint_results_df["evaluated_epoch"], checkpoint_results_df[col], marker="o", label=col)
    ax.set_title(title)
    ax.set_ylabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("evaluated epoch")
plt.tight_layout()
plot_path = os.path.join(run_dir, "checkpoint_evaluation_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

per_class_cols = [col for col in checkpoint_results_df.columns if col.startswith("recall_")]
if len(per_class_cols):
    fig, axes = plt.subplots(3, 1, figsize=(12, max(10, len(per_class_cols) * 0.45)), sharex=True)
    split_names = ["train", "val", "test"]
    for ax, split_name in zip(axes, split_names):
        split_cols = [col for col in per_class_cols if col.endswith(f"_{split_name}")]
        for col in split_cols:
            ax.plot(checkpoint_results_df["evaluated_epoch"], checkpoint_results_df[col], marker="o", label=col)
        ax.set_title(f"Per-class Recall ({split_name})")
        ax.set_ylabel("recall")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)
    axes[-1].set_xlabel("evaluated epoch")
    plt.tight_layout()
    per_class_plot_path = os.path.join(run_dir, "checkpoint_recall_plot.png")
    plt.savefig(per_class_plot_path, dpi=150, bbox_inches="tight")
    plt.show()

checkpoint_results_df
